# How to fine-tune chat models

Fine-tuning improves the model by training on many more examples than can fit in a prompt, letting you achieve better results on a wide number of tasks. This notebook provides a step-by-step guide for our new GPT-4o mini fine-tuning. We'll perform entity extraction using the [RecipeNLG dataset](https://github.com/Glorf/recipenlg), which provides various recipes and a list of extracted generic ingredients for each. This is a common dataset for named entity recognition (NER) tasks.

Note: **GPT-4o mini fine-tuning is available to developers in our [Tier 4 and 5 usage tiers](https://platform.openai.com/docs/guides/rate-limits/usage-tiers).** You can start fine-tuning GPT-4o mini by visiting your fine-tuning dashboard, clicking "create", and selecting “gpt-4o-mini-2024-07-18” from the base model drop-down.

We will go through the following steps:

1. **Setup:** Loading our dataset and filtering down to one domain to fine-tune on.
2. **Data preparation:** Preparing your data for fine-tuning by creating training and validation examples, and uploading them to the `Files` endpoint.
3. **Fine-tuning:** Creating your fine-tuned model.
4. **Inference:** Using your fine-tuned model for inference on new inputs.

By the end of this you should be able to train, evaluate and deploy a fine-tuned `gpt-4o-mini-2024-07-18` model.

For more information on fine-tuning, you can refer to our [documentation guide](https://platform.openai.com/docs/guides/fine-tuning) or [API reference](https://platform.openai.com/docs/api-reference/fine-tuning).


## Setup


In [1]:
# make sure to use the latest version of the openai python package
!pip install --upgrade --quiet openai


[notice] A new release of pip is available: 24.0 -> 25.0
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import openai
import os
import pandas as pd
from pprint import pprint
from sklearn.model_selection import train_test_split

client = openai.OpenAI(api_key=os.environ.get(
    "OPENAI_API_KEY"))

Fine-tuning works best when focused on a particular domain. It's important to make sure your dataset is both focused enough for the model to learn, but general enough that unseen examples won't be missed. Having this in mind, we have extracted a subset from the RecipesNLG dataset to only contain documents from [cookbooks.com](https://cookbooks.com/).


In [3]:
# Read in the dataset we'll use for this task.
# This will be the RecipesNLG dataset, which we've cleaned to only contain documents from www.cookbooks.com
df = pd.read_csv("few_shots_data.csv")


print(df.shape)
df.head()

(3860, 4)


,sentence1,sentence2,idx,label
0,use of healing resources,providing healing resources for police,0,0
1,use of healing resources,need for healing,1,0
2,use of healing resources,trust in healing resources,2,0
3,use of healing resources,cultural competency of healing resources,3,0
4,use of healing resources,availability of healing resources for trauma,4,0


## Data preparation

We'll begin by preparing our data. When fine-tuning with the `ChatCompletion` format, each training example is a simple list of `messages`. For example, an entry could look like:

```
[{'role': 'system',
  'content': 'You are a helpful recipe assistant. You are to extract the generic ingredients from each of the recipes provided.'},

 {'role': 'user',
  'content': 'Title: No-Bake Nut Cookies\n\nIngredients: ["1 c. firmly packed brown sugar", "1/2 c. evaporated milk", "1/2 tsp. vanilla", "1/2 c. broken nuts (pecans)", "2 Tbsp. butter or margarine", "3 1/2 c. bite size shredded rice biscuits"]\n\nGeneric ingredients: '},

 {'role': 'assistant',
  'content': '["brown sugar", "milk", "vanilla", "nuts", "butter", "bite size shredded rice biscuits"]'}]
```

During the training process this conversation will be split, with the final entry being the `completion` that the model will produce, and the remainder of the `messages` acting as the prompt. Consider this when building your training examples - if your model will act on multi-turn conversations, then please provide representative examples so it doesn't perform poorly when the conversation starts to expand.

Please note that currently there is a 4096 token limit for each training example. Anything longer than this will be truncated at 4096 tokens.


In [5]:
system_message = """

You are a helpful assistant. You will be given two sentencs, and you will have to label the semantic relationship between them. 
If the two sentences are semantically related, you will label them as 1. If they are not semantically related, you will label them as 0. 

"""


def create_user_message(row):
    return f"Sentence: {row['sentence1']}\n\nSentence: {row['sentence2']}\n"


def prepare_example_conversation(row):
    return {
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": create_user_message(row)},
            {"role": "assistant", "content": row["label"]},
        ]
    }


pprint(prepare_example_conversation(df.iloc[0]))

{'messages': [{'content': '\n'
                          '\n'
                          'You are a helpful assistant. You will be given two '
                          'sentencs, and you will have to label the semantic '
                          'relationship between them. \n'
                          'If the two sentences are semantically related, you '
                          'will label them as 1. If they are not semantically '
                          'related, you will label them as 0. \n'
                          '\n',
               'role': 'system'},
              {'content': 'Sentence: use of healing resources\n'
                          '\n'
                          'Sentence: providing healing resources for police\n',
               'role': 'user'},
              {'content': '0', 'role': 'assistant'}]}


Let's now do this for a subset of the dataset to use as our training data. You can begin with even 30-50 well-pruned examples. You should see performance continue to scale linearly as you increase the size of the training set, but your jobs will also take longer.


In [7]:
# delete rows when label is empty
print("before dropna", df.shape)
df = df.dropna(subset=['label'])
print("after dropna", df.shape)
# split train and test data

training_df, t_df = train_test_split(
    df, test_size=0.2, random_state=42)

validation_df, test_df = train_test_split(t_df, test_size=0.5, random_state=42)

print("train df", training_df.shape)
print("validation df", validation_df.shape)
print("test df", test_df.shape)

before dropna (3860, 4)
after dropna (2486, 4)
train df (1988, 4)
validation df (249, 4)
test df (249, 4)


In [82]:
# use the first 100 rows of the dataset for training
# training_df = df.loc[0:100]

# apply the prepare_example_conversation function to each row of the training_df
training_data = training_df.apply(
    prepare_example_conversation, axis=1).tolist()

for example in training_data[:5]:
    print(example)

{'messages': [{'role': 'system', 'content': '\n\nYou are a helpful assistant. You will be given two sentencs, and you will have to label the semantic relationship between them. \nIf the two sentences are semantically related, you will label them as 1. If they are not semantically related, you will label them as 0. \n\n'}, {'role': 'user', 'content': 'Sentence: failure to protect other family members\n\nSentence: family enforced silence of talking about violence\n'}, {'role': 'assistant', 'content': '0'}]}
{'messages': [{'role': 'system', 'content': '\n\nYou are a helpful assistant. You will be given two sentencs, and you will have to label the semantic relationship between them. \nIf the two sentences are semantically related, you will label them as 1. If they are not semantically related, you will label them as 0. \n\n'}, {'role': 'user', 'content': 'Sentence: people reporting an incident of intimate partner violence to the police\n\nSentence: children witness intimate partner violenc

In addition to training data, we can also **optionally** provide validation data, which will be used to make sure that the model does not overfit your training set.


In [83]:
# validation_df = df.loc[101:200]
validation_data = validation_df.apply(
    prepare_example_conversation, axis=1).tolist()

We then need to save our data as `.jsonl` files, with each line being one training example conversation.


In [85]:
def write_jsonl(data_list: list, filename: str) -> None:
    with open(filename, "w") as out:
        for ddict in data_list:
            jout = json.dumps(ddict) + "\n"
            out.write(jout)

In [86]:
training_file_name = "tmp_recipe_finetune_training.jsonl"
write_jsonl(training_data, training_file_name)

validation_file_name = "tmp_recipe_finetune_validation.jsonl"
write_jsonl(validation_data, validation_file_name)

This is what the first 5 lines of our training `.jsonl` file look like:


In [87]:
# print the first 5 lines of the training file
!head -n 5 tmp_recipe_finetune_training.jsonl

{"messages": [{"role": "system", "content": "\n\nYou are a helpful assistant. You will be given two sentencs, and you will have to label the semantic relationship between them. \nIf the two sentences are semantically related, you will label them as 1. If they are not semantically related, you will label them as 0. \n\n"}, {"role": "user", "content": "Sentence: failure to protect other family members\n\nSentence: family enforced silence of talking about violence\n"}, {"role": "assistant", "content": "0"}]}
{"messages": [{"role": "system", "content": "\n\nYou are a helpful assistant. You will be given two sentencs, and you will have to label the semantic relationship between them. \nIf the two sentences are semantically related, you will label them as 1. If they are not semantically related, you will label them as 0. \n\n"}, {"role": "user", "content": "Sentence: people reporting an incident of intimate partner violence to the police\n\nSentence: children witness intimate partner violenc

### Upload files

You can now upload the files to our `Files` endpoint to be used by the fine-tuned model.


In [88]:
def upload_file(file_name: str, purpose: str) -> str:
    with open(file_name, "rb") as file_fd:
        response = client.files.create(file=file_fd, purpose=purpose)
    return response.id


training_file_id = upload_file(training_file_name, "fine-tune")
validation_file_id = upload_file(validation_file_name, "fine-tune")

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)

Training file ID: file-4awU8b5wGqWBgokoSzr5TG
Validation file ID: file-JMti7EidLtMVrwzmCCwkD5


## Fine-tuning

Now we can create our fine-tuning job with the generated files and an optional suffix to identify the model. The response will contain an `id` which you can use to retrieve updates on the job.

Note: The files have to first be processed by our system, so you might get a `File not ready` error. In that case, simply retry a few minutes later.


In [89]:
MODEL = "gpt-4o-mini-2024-07-18"

response = client.fine_tuning.jobs.create(
    training_file=training_file_id,
    validation_file=validation_file_id,
    model=MODEL,
    suffix="recipe-ner",
)

job_id = response.id

print("Job ID:", response.id)
print("Status:", response.status)

Job ID: ftjob-dttfKWZEbLqmzRJXLXtaYEbM
Status: validating_files


#### Check job status

You can make a `GET` request to the `https://api.openai.com/v1/alpha/fine-tunes` endpoint to list your alpha fine-tune jobs. In this instance you'll want to check that the ID you got from the previous step ends up as `status: succeeded`.

Once it is completed, you can use the `result_files` to sample the results from the validation set (if you uploaded one), and use the ID from the `fine_tuned_model` parameter to invoke your trained model.


In [90]:
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)

Job ID: ftjob-dttfKWZEbLqmzRJXLXtaYEbM
Status: succeeded
Trained Tokens: 559461


We can track the progress of the fine-tune with the events endpoint. You can rerun the cell below a few times until the fine-tune is ready.


In [91]:
# job_id = "ftjob-v3vNEZmjMFFimOy9TNcFrI7U"

In [92]:
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)

Step 1973/1988: training loss=0.00
Step 1974/1988: training loss=0.00
Step 1975/1988: training loss=0.00
Step 1976/1988: training loss=0.00
Step 1977/1988: training loss=0.00
Step 1978/1988: training loss=0.00
Step 1979/1988: training loss=0.00
Step 1980/1988: training loss=0.00
Step 1981/1988: training loss=0.00
Step 1982/1988: training loss=0.00
Step 1983/1988: training loss=0.00
Step 1984/1988: training loss=0.00
Step 1985/1988: training loss=0.00
Step 1986/1988: training loss=0.00
Step 1987/1988: training loss=0.00
Step 1988/1988: training loss=0.00, validation loss=0.00, full validation loss=0.30
Checkpoint created at step 663
Checkpoint created at step 1326
New fine-tuned model created
The job has successfully completed


Now that it's done, we can get a fine-tuned model ID from the job:


In [93]:
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model

if fine_tuned_model_id is None:
    raise RuntimeError(
        "Fine-tuned model ID not found. Your job has likely not been completed yet."
    )

print("Fine-tuned model ID:", fine_tuned_model_id)

Fine-tuned model ID: ft:gpt-4o-mini-2024-07-18:personal:recipe-ner:Ay8kQrV0


## Inference


The last step is to use your fine-tuned model for inference. Similar to the classic `FineTuning`, you simply call `ChatCompletions` with your new fine-tuned model name filling the `model` parameter.


In [94]:
test_df = df.loc[201:300]
test_row = test_df.iloc[8]
test_messages = []
test_messages.append({"role": "system", "content": system_message})
user_message = create_user_message(test_row)
test_messages.append({"role": "user", "content": user_message})

pprint(test_messages)

[{'content': '\n'
             '\n'
             'You are a helpful assistant. You will be given two sentencs, and '
             'you will have to label the semantic relationship between them. \n'
             'If the two sentences are semantically related, you will label '
             'them as 1. If they are not semantically related, you will label '
             'them as 0. \n'
             '\n',
  'role': 'system'},
 {'content': 'Sentence: trust in police\n\nSentence: fear of police\n',
  'role': 'user'}]


In [95]:
response = client.chat.completions.create(
    model=fine_tuned_model_id, messages=test_messages, temperature=0, max_tokens=500
)
print(response.choices[0].message.content)

0


## Conclusion

Congratulations, you are now ready to fine-tune your own models using the `ChatCompletion` format! We look forward to seeing what you build


In [96]:
test_df = test_df.reset_index(drop=True)

In [97]:
final_results = []

for index, row in test_df.iterrows():

    test_row = test_df.iloc[index]

    test_messages = []
    test_messages.append({"role": "system", "content": system_message})
    user_message = create_user_message(test_row)

    test_messages.append({"role": "user", "content": user_message})

    # pprint(test_messages)

    response = client.chat.completions.create(
        model=fine_tuned_model_id, messages=test_messages, temperature=0, max_tokens=500
    )
    # print(response.choices[0].message.content)
    final_results.append(response.choices[0].message.content)

test_df["predicted_label"] = final_results

In [99]:
test_df.head()

,sentence1,sentence2,idx,label,predicted_label
0,personal income,family income,201,0,0
1,ability to acquire a job,attaining job,202,0,0
2,ability to acquire a job,employment,203,0,0
3,homelessness,incarceration,204,0,0
4,culture of silence,suppression of culture,205,0,0
...,...,...,...,...,...
95,community level alcohol use,sobriety in social group,296,0,0
96,community level alcohol use,youth alcohol use,297,0,0
97,community level alcohol use,alcohol misuse,298,0,0
98,community level alcohol use,community alcohol use,299,1,1


## Result: nice! accuracy is 0.93!

In [100]:
# calculate the accruacy of the model on the test set
import numpy as np

from sklearn.metrics import accuracy_score
y_pred = final_results
y_true = np.array(test_df['label'])

print(accuracy_score(y_true, y_pred))

0.93


# predict the rest of the data

In [8]:
job_id = "ftjob-dttfKWZEbLqmzRJXLXtaYEbM"

In [9]:
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model

if fine_tuned_model_id is None:
    raise RuntimeError(
        "Fine-tuned model ID not found. Your job has likely not been completed yet."
    )

print("Fine-tuned model ID:", fine_tuned_model_id)

Fine-tuned model ID: ft:gpt-4o-mini-2024-07-18:personal:recipe-ner:Ay8kQrV0


In [10]:
test_mes = [{'content': '\n'
             '\n'
             'You are a helpful assistant. You will be given two sentencs, and '
             'you will have to label the semantic relationship between them. \n'
             'If the two sentences are semantically related, you will label '
             'them as 1. If they are not semantically related, you will label '
             'them as 0. \n'
             '\n',
             'role': 'system'},
            {'content': 'Sentence: trust in police\n\nSentence: fear of police\n',
             'role': 'user'}]

response = client.chat.completions.create(
    model=fine_tuned_model_id, messages=test_mes, temperature=0, max_tokens=500
)
print(response.choices[0].message.content)

0


In [6]:
# all_variables_pairs_df = pd.read_csv("all_variable_pairs.csv")
all_variables_pairs_df = pd.read_csv("all_definition_pairs.csv")


all_variables_pairs_df.rename(
    columns={"Variable 1": "sentence1", "Variable 2": "sentence2"}, inplace=True)
all_variables_pairs_df.head()

all_variables_pairs_df = all_variables_pairs_df.loc[0:50]

In [10]:
final_results = []

for index, row in all_variables_pairs_df.iterrows():

    test_row = all_variables_pairs_df.iloc[index]

    test_messages = []
    test_messages.append({"role": "system", "content": system_message})
    user_message = create_user_message(test_row)

    test_messages.append({"role": "user", "content": user_message})

    # pprint(test_messages)

    response = client.chat.completions.create(
        model=fine_tuned_model_id, messages=test_messages, temperature=0, max_tokens=500
    )
    # print(response.choices[0].message.content)
    final_results.append(response.choices[0].message.content)

all_variables_pairs_df["predicted_label"] = final_results

In [11]:
all_variables_pairs_df

,sentence1,sentence2,predicted_label
0,using alcohol - in reference to misuse,perpetration or victimization of violence by a...,0
1,using alcohol - in reference to misuse,"""Increases"" in mental health indicate a declin...",0
2,using alcohol - in reference to misuse,the occurrence of a (physical mostly) violent ...,0
3,using alcohol - in reference to misuse,perpetration and or victimization of violence ...,0
4,using alcohol - in reference to misuse,reporting an IPV event to the police,0
5,using alcohol - in reference to misuse,discriminatory treatment that Indigenous peopl...,0
6,using alcohol - in reference to misuse,the policies and practices that covertly maint...,0
7,using alcohol - in reference to misuse,police physical violence against indigenous pe...,0
8,using alcohol - in reference to misuse,people getting involved with the police throug...,0
9,using alcohol - in reference to misuse,how helpful or positive interactions with pol...,0
